# NER Evaluation: Ground Truth vs Predictions
Run the cells **from top to bottom** (Shift+Enter). Put this notebook in the project root, next to `make_excel.py`. Close both Excel files first.

## Step 0: install openpyxl (only needed once)
If you already installed it, skip this cell.

In [ ]:
%pip install openpyxl

## Step 1: settings
Paths and the label mapping. Nothing to change unless you disagree with a choice.

In [ ]:
import re
import datetime as dt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 200)

GT_PATH = "results/groundTruth_20.xlsx"
PRED_PATH = "results/predictions_20.xlsx"

# Team labels (your GT uses ORG for ORGANIZATION)
LABELS = ["PERSON", "ORG", "LOCATION", "DATE", "PRODUCT", "REFERENCE_NUMBER"]

# spaCy label -> team label. Applied to PREDICTIONS only, so they can be compared.
# Labels not listed here are "out of scheme" and are reported separately.
PRED_LABEL_MAP = {
    "PERSON": "PERSON",
    "ORG": "ORG",
    "GPE": "LOCATION",
    "LOC": "LOCATION",
    "DATE": "DATE",
    "TIME": "DATE",     # GT counts "Tuesday at 11:45" as one DATE
    "PRODUCT": "PRODUCT",
}

## Step 2: load both files and clean the text
Cleaning only happens in memory. Your Excel files are not changed.

In [ ]:
def clean_text(x):
    """Dates Excel turned into datetime -> '14 may 2001'. Cuts junk like '\\nX-cc'. Lowercases."""
    if isinstance(x, (dt.datetime, dt.date, pd.Timestamp)):
        x = f"{x.day} {x.strftime('%b %Y')}"
    x = str(x)
    x = re.split(r"\\n|\n", x)[0]
    x = re.sub(r"\s+", " ", x.lower()).strip(" .,:;")
    return x

def load(path, is_pred):
    df = pd.read_excel(path)
    df["label_raw"] = df["label"].astype(str).str.strip()
    df["label"] = df["label_raw"].map(PRED_LABEL_MAP) if is_pred else df["label_raw"]
    df["t"] = df["text"].map(clean_text)
    return df

gt = load(GT_PATH, is_pred=False)
pred_all = load(PRED_PATH, is_pred=True)

print("GT rows:", len(gt), "| documents:", gt.document_id.nunique())
print("Pred rows:", len(pred_all), "| documents:", pred_all.document_id.nunique())

## Step 3: look at the data
Count of entities per label in each file.

In [ ]:
counts = pd.DataFrame({
    "ground_truth": gt.label_raw.value_counts(),
    "predictions (raw spaCy labels)": pred_all.label_raw.value_counts(),
}).fillna(0).astype(int)
display(counts)

## Step 4: prepare the comparison
The GT lists each entity **once per email**, but the model lists **every mention**, so we compare distinct entities per email and label.

In [ ]:
out_of_scheme = pred_all[pred_all["label"].isna()]
pred = pred_all.dropna(subset=["label"])

gt_set = gt[gt.t != ""].drop_duplicates(["document_id", "label", "t"])
pred_set = pred[pred.t != ""].drop_duplicates(["document_id", "label", "t"])

print("Distinct GT entities:", len(gt_set))
print("Distinct predicted entities (in scheme):", len(pred_set))
print("Out-of-scheme predictions (not scored):", len(out_of_scheme))

## Step 5: score it
- **strict**: exact text and same label
- **relaxed**: same label, and one text contains the other

In [ ]:
def contains(a, b):
    return a in b or b in a

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return round(p, 3), round(r, 3), round(f, 3)

rows, errors = [], []
for mode in ["strict", "relaxed"]:
    TP = FP = FN = 0
    for lab in LABELS:
        tp = fp = fn = 0
        for doc in sorted(gt_set.document_id.unique()):
            g = set(gt_set[(gt_set.document_id == doc) & (gt_set.label == lab)].t)
            p = set(pred_set[(pred_set.document_id == doc) & (pred_set.label == lab)].t)
            if mode == "strict":
                m_g, m_p = g & p, g & p
            else:
                m_g = {x for x in g if any(contains(x, y) for y in p)}
                m_p = {y for y in p if any(contains(x, y) for x in g)}
            tp += len(m_g); fn += len(g - m_g); fp += len(p - m_p)
            if mode == "strict":
                errors += [{"doc": doc, "label": lab, "type": "FN (missed)", "entity": x} for x in g - m_g]
                errors += [{"doc": doc, "label": lab, "type": "FP (extra)", "entity": x} for x in p - m_p]
        TP += tp; FP += fp; FN += fn
        pr, rc, f1 = prf(tp, fp, fn)
        rows.append({"mode": mode, "label": lab, "precision": pr, "recall": rc, "f1": f1, "tp": tp, "fp": fp, "fn": fn})
    pr, rc, f1 = prf(TP, FP, FN)
    rows.append({"mode": mode, "label": "OVERALL (micro)", "precision": pr, "recall": rc, "f1": f1, "tp": TP, "fp": FP, "fn": FN})

summary = pd.DataFrame(rows)
errors_df = pd.DataFrame(errors)
print("Done.")

## Step 6: results table

In [ ]:
for mode in ["strict", "relaxed"]:
    print(f"\n=== {mode.upper()} ===")
    display(summary[summary["mode"] == mode].drop(columns="mode").reset_index(drop=True))

## Step 7: chart (F1 per label, strict vs relaxed)

In [ ]:
import matplotlib.pyplot as plt

pivot = summary.pivot(index="label", columns="mode", values="f1").loc[LABELS + ["OVERALL (micro)"]]
ax = pivot.plot(kind="bar", figsize=(9, 4), ylim=(0, 1))
ax.set_ylabel("F1")
ax.set_title("F1 per label: strict vs relaxed")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Step 8: errors
How many errors per label, then the actual list.
- **FN (missed)**: in your GT, the model did not find it
- **FP (extra)**: the model found it, your GT does not have it

In [ ]:
display(errors_df.groupby(["label", "type"]).size().unstack(fill_value=0))

In [ ]:
# Change this to look at another label: PERSON, ORG, LOCATION, DATE, PRODUCT, REFERENCE_NUMBER
LABEL_TO_SHOW = "ORG"

display(errors_df[errors_df.label == LABEL_TO_SHOW].sort_values(["type", "doc"]).reset_index(drop=True))

## Step 9: predictions with no team label (not scored)

In [ ]:
display(out_of_scheme.label_raw.value_counts().to_frame("count"))
display(out_of_scheme[["document_id", "text", "label_raw"]].head(20))

## Step 10: save the results to Excel

In [ ]:
summary.to_excel("results/evaluation_summary.xlsx", index=False)
errors_df.to_excel("results/errors.xlsx", index=False)
out_of_scheme[["document_id", "text", "label_raw"]].to_excel("results/out_of_scheme_predictions.xlsx", index=False)
print("Saved in results/: evaluation_summary.xlsx, errors.xlsx, out_of_scheme_predictions.xlsx")